In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from modules import util

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Load Dataset

In [2]:
housing = fetch_california_housing(as_frame=True)
feature_names = housing.feature_names
target_names = housing.target_names
data = housing.frame

In [3]:
data

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


## Train-Val-Test Splitting

In [4]:
train, test = util.split_train_test_masks(list(data.index), train_pct=.7, test_pct=.3, seed=42)
data[['TrainMask', 'TestMask']] = np.column_stack((train, test))
data

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,TrainMask,TestMask
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,1,0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,0,1
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,1,0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,0,1
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,0,1
...,...,...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781,1,0
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771,0,1
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923,0,1
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847,1,0


# Traditional ML Methods

In [5]:
X_train = data.loc[data.TrainMask == 1, feature_names].values
y_train = data.loc[data.TrainMask == 1, target_names[0]]
X_test = data.loc[data.TestMask == 1, feature_names].values
y_test = data.loc[data.TestMask == 1, target_names[0]]

## Model Selection

In [6]:
from modules.model_search import halving_grid_search_models_with_test
import warnings
warnings.filterwarnings('ignore')

results = halving_grid_search_models_with_test(X_train, X_test, y_train, y_test, cv=5, verbose=3, n_jobs=-1, random_state=42)

n_iterations: 7
n_required_iterations: 7
n_possible_iterations: 7
min_resources_: 19
max_resources_: 14448
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 1280
n_resources: 19
Fitting 5 folds for each of 1280 candidates, totalling 6400 fits


/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/li

[CV 2/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=uniform;, score=-0.710 total time=   0.0s
[CV 2/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=distance;, score=-0.710 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=distance;, score=-3.457 total time=   0.0s
[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-3.808 total time=   0.0s[CV 5/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=uniform;, score=-1.800 total time=   0.0s

[CV 1/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=uniform;, score=-8.907 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=2, model__weights=unif

/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/li

[CV 5/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=uniform;, score=-1.800 total time=   0.0s
[CV 2/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=uniform;, score=-0.710 total time=   0.0s
[CV 1/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=uniform;, score=-8.907 total time=   0.0s
[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-3.767 total time=   0.0s
[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=1, model__weights=uniform;, score=-3.843 total time=   0.0s
[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=uniform;, score=-4.660 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=2, model__weights=dista

/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/li

[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=5, model__p=1, model__weights=distance;, score=-2.339 total time=   0.0s
[CV 2/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=distance;, score=-0.710 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=5, model__p=1, model__weights=distance;, score=-0.881 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=distance;, score=-2.999 total time=   0.0s
[CV 5/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=5, model__p=1, model__weights=distance;, score=-0.618 total time=   0.0s
[CV 1/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=1, model__p=2, model__weights=distance;, score=-8.907 total time=   0.0s
[CV 1/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=5, model__p=2, model__weights=

/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/lib64/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/li

[CV 1/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=7, model__p=1, model__weights=uniform;, score=-1.483 total time=   0.0s
[CV 2/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=7, model__p=1, model__weights=uniform;, score=-0.657 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=1, model__weights=distance;, score=-1.217 total time=   0.0s
[CV 3/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=7, model__p=1, model__weights=uniform;, score=-1.550 total time=   0.0s
[CV 5/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=3, model__p=1, model__weights=distance;, score=-1.506 total time=   0.0s
[CV 4/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=7, model__p=1, model__weights=uniform;, score=-0.781 total time=   0.0s
[CV 5/5] END model__algorithm=auto, model__leaf_size=10, model__n_neighbors=7, model__p=1, model__weights=unif

KeyboardInterrupt: 

In [ ]:
results